In [0]:
orders_raw = spark.table("emil_cat.bronze.orders_raw")
customers_raw = spark.table("emil_cat.bronze.customers_raw")


In [0]:
customers_lookup = customers_raw.select("customer_id").dropDuplicates(["customer_id"])

orders_clean = orders_raw.join(customers_lookup, on="customer_id", how="inner")
orders_quarantine = orders_raw.join(customers_lookup, on="customer_id", how="left_anti")

print("Clean orders:", orders_clean.count())
print("Quarantined orders:", orders_quarantine.count())

In [0]:
from pyspark.sql import functions as F

customers_lookup = (
    customers_raw.select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumnRenamed("customer_id", "lookup_customer_id")
)

orders_checked = orders_raw.join(
    customers_lookup,
    orders_raw.customer_id == customers_lookup.lookup_customer_id,
    how="left"
)

orders_checked = (
    orders_checked
    .withColumn("is_valid_customer", F.col("lookup_customer_id").isNotNull())
    .withColumn("is_valid_quantity", F.col("quantity").isNotNull())
    .withColumn(
        "quarantine_reason",
        F.when(~F.col("is_valid_customer") & ~F.col("is_valid_quantity"), "invalid_customer_and_null_quantity")
         .when(~F.col("is_valid_customer"), "invalid_customer_id")
         .when(~F.col("is_valid_quantity"), "null_quantity")
    )
)

orders_clean = (
    orders_checked
    .filter(F.col("is_valid_customer") & F.col("is_valid_quantity"))
    .drop("lookup_customer_id", "is_valid_customer", "is_valid_quantity", "quarantine_reason")
)

orders_quarantine = (
    orders_checked
    .filter(~(F.col("is_valid_customer") & F.col("is_valid_quantity")))
    .drop("lookup_customer_id", "is_valid_customer", "is_valid_quantity")
)

print("Clean:", orders_clean.count())
print("Quarantined:", orders_quarantine.count())
display(orders_quarantine)

In [0]:
orders_clean.write.mode("overwrite").saveAsTable("emil_cat.silver.orders_clean")
orders_quarantine.write.mode("overwrite").saveAsTable("emil_cat.silver.orders_quarantine")

In [0]:
from pyspark.sql import functions as F

customers_raw = spark.table("emil_cat.bronze.customers_raw")

customers_clean = (
    customers_raw
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

customers_quarantine = customers_raw.filter(F.col("customer_id").isNull())

print("Clean:", customers_clean.count())
print("Quarantined:", customers_quarantine.count())

customers_clean.write.mode("overwrite").saveAsTable("emil_cat.silver.customers_clean")
customers_quarantine.write.mode("overwrite").saveAsTable("emil_cat.silver.customers_quarantine")

In [0]:
from pyspark.sql import functions as F

products_raw = spark.table("emil_cat.bronze.products_raw")

products_clean = (
    products_raw
    .withColumn("product_code", F.upper(F.trim(F.col("product_code"))))
    .dropDuplicates(["product_code"])
)

print("Original rows:", products_raw.count())
print("Clean rows:", products_clean.count())

products_clean.write.mode("overwrite").saveAsTable("emil_cat.silver.products_clean")
display(products_clean)